# Notebook 04 — Modelling and Evaluation

## Objectives
This notebook addresses **Business Requirement 2** — predicting whether a customer will churn. I build a full ML pipeline, tune hyperparameters, and use decision threshold tuning on the validation set to meet the ≥75% recall target before doing a final evaluation on the held-out test set.

* Split data into train, validation, and test sets
* Build a preprocessing + classifier pipeline with SMOTE oversampling
* Search for the best GradientBoostingClassifier hyperparameters using GridSearchCV
* Tune the decision threshold on the validation set to achieve ≥75% recall
* Evaluate final performance on the test set using the CI-style confusion matrix and classification report
* Save the fitted pipeline and optimal threshold

## Inputs
* `outputs/datasets/collection/telco_churn_cleaned.csv`

## Outputs
* `outputs/ml_pipeline/predict_churn/v1/clf_pipeline.pkl`
* `outputs/ml_pipeline/predict_churn/v1/optimal_threshold.pkl`
* `outputs/ml_pipeline/predict_churn/v1/confusion_matrix.png`
* `outputs/ml_pipeline/predict_churn/v1/threshold_tuning.png`
* `outputs/ml_pipeline/predict_churn/v1/classification_report.csv`

---

## 1 — Set Up the Working Directory

In [ ]:
import os

os.chdir('..') if os.path.basename(os.getcwd()) == 'jupyter_notebooks' else None

PIPELINE_DIR = 'outputs/ml_pipeline/predict_churn/v1'
os.makedirs(PIPELINE_DIR, exist_ok=True)
print('Working directory:', os.getcwd())

## 2 — Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import joblib

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.metrics import (
    classification_report, confusion_matrix,
    ConfusionMatrixDisplay, precision_recall_curve
)
from sklearn.ensemble import GradientBoostingClassifier
from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline as ImbPipeline

## 3 — Load the Cleaned Dataset

In [ ]:
df = pd.read_csv('outputs/datasets/collection/telco_churn_cleaned.csv')
print(f'Loaded: {df.shape[0]} rows, {df.shape[1]} columns')
df.head(3)

## 4 — Define Features and Target

In [ ]:
TARGET = 'Churn'
X = df.drop(columns=[TARGET])
y = df[TARGET]

CATEGORICAL_FEATURES = X.select_dtypes(include='object').columns.tolist()
NUMERIC_FEATURES = X.select_dtypes(include=[np.number]).columns.tolist()

print('Categorical features:', CATEGORICAL_FEATURES)
print('Numeric features:', NUMERIC_FEATURES)
print(f'\nChurn rate: {y.mean()*100:.1f}%')

## 5 — Train / Validation / Test Split

I use a three-way split:
- **Train (60%)** — used to fit the model
- **Validation (20%)** — used to tune the decision threshold without touching the test set
- **Test (20%)** — held out completely until the very end for final evaluation

All splits are stratified to preserve the original churn ratio.

In [ ]:
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.4, random_state=42, stratify=y
)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.5, random_state=42, stratify=y_temp
)

print(f'Train : {X_train.shape[0]} rows | Churn rate: {y_train.mean()*100:.1f}%')
print(f'Val   : {X_val.shape[0]} rows  | Churn rate: {y_val.mean()*100:.1f}%')
print(f'Test  : {X_test.shape[0]} rows  | Churn rate: {y_test.mean()*100:.1f}%')

## 6 — Helper Functions (CI Style)

I am using the Code Institute evaluation helper functions, adapted for a sklearn GradientBoosting pipeline with a configurable probability threshold.

In [ ]:
def confusion_matrix_and_report(X, y, pipeline, label_map, threshold=0.5):
    y_proba = pipeline.predict_proba(X)[:, 1]
    prediction = (y_proba >= threshold).astype(int)

    print('---  Confusion Matrix  ---')
    print(pd.DataFrame(
        confusion_matrix(y_true=y, y_pred=prediction),
        columns=['Actual ' + sub for sub in label_map],
        index=['Predicted ' + sub for sub in label_map]
    ))
    print('\n')
    print('---  Classification Report  ---')
    print(classification_report(y, prediction, target_names=label_map), '\n')


def clf_performance(X_train, y_train, X_val, y_val, X_test, y_test,
                    pipeline, label_map, threshold=0.5):
    print('#### Train Set ####\n')
    confusion_matrix_and_report(X_train, y_train, pipeline, label_map, threshold)

    print('#### Validation Set ####\n')
    confusion_matrix_and_report(X_val, y_val, pipeline, label_map, threshold)

    print('#### Test Set ####\n')
    confusion_matrix_and_report(X_test, y_test, pipeline, label_map, threshold)

## 7 — Build the ML Pipeline

The pipeline has three stages:
1. **Preprocessing** — OneHotEncoder for categorical features, StandardScaler for numeric features
2. **SMOTE** — oversamples the minority class (Churn = 1) within the training fold only to prevent leakage
3. **GradientBoostingClassifier** — a boosting ensemble that iteratively corrects errors from previous trees, making it effective at identifying minority class patterns

In [ ]:
preprocessor = ColumnTransformer(transformers=[
    ('cat', OneHotEncoder(handle_unknown='ignore', drop='first'), CATEGORICAL_FEATURES),
    ('num', StandardScaler(), NUMERIC_FEATURES),
])

clf_pipeline = ImbPipeline(steps=[
    ('preprocessor', preprocessor),
    ('smote', SMOTE(random_state=42, k_neighbors=5)),
    ('classifier', GradientBoostingClassifier(random_state=42)),
])

print('Pipeline defined:')
print(clf_pipeline)

## 8 — Hyperparameter Search

I use GridSearchCV with 5-fold cross-validation, optimising for **recall** on the Churn class.

The parameter grid is based on the Code Institute reference notebook for GradientBoostingClassifier.

In [ ]:
params_search = {
    'classifier__n_estimators': [100, 50, 140],
    'classifier__learning_rate': [0.1, 0.01, 0.001],
    'classifier__max_depth': [3, 15, None],
    'classifier__min_samples_split': [2, 50],
    'classifier__min_samples_leaf': [1, 50],
    'classifier__max_leaf_nodes': [None, 50],
}

grid_search = GridSearchCV(
    clf_pipeline,
    param_grid=params_search,
    cv=5,
    scoring='recall',
    n_jobs=-1,
    verbose=1,
)

grid_search.fit(X_train, y_train)

print('\nBest parameters found:')
print(grid_search.best_params_)
print(f'Best CV recall score: {grid_search.best_score_:.4f}')

## 9 — Decision Threshold Tuning on Validation Set

The default prediction threshold of 0.5 means we only flag a customer as churner if the model is more than 50% confident. This is too conservative for our use case — missing a churner is more costly than a false alarm.

I use the **precision-recall curve on the validation set** to find the highest threshold that still achieves ≥75% recall. Tuning on the validation set rather than the test set keeps the test set completely unseen.

In [ ]:
best_pipeline = grid_search.best_estimator_

y_proba_val = best_pipeline.predict_proba(X_val)[:, 1]
precisions, recalls, thresholds = precision_recall_curve(y_val, y_proba_val)

fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(thresholds, precisions[:-1], label='Precision', color='steelblue')
ax.plot(thresholds, recalls[:-1], label='Recall', color='tomato')
ax.axhline(y=0.75, color='green', linestyle='--', label='75% Recall target')
ax.set_xlabel('Decision Threshold')
ax.set_ylabel('Score')
ax.set_title('Precision vs Recall Trade-off (Validation Set)', fontsize=13, fontweight='bold')
ax.legend()
ax.set_xlim([0, 1])
ax.set_ylim([0, 1])
plt.tight_layout()
plt.savefig(os.path.join(PIPELINE_DIR, 'threshold_tuning.png'), dpi=150)
plt.show()

TARGET_RECALL = 0.75
valid_indices = np.where(recalls[:-1] >= TARGET_RECALL)[0]

if len(valid_indices) > 0:
    optimal_idx = valid_indices[-1]
    optimal_threshold = float(thresholds[optimal_idx])
    print(f'Optimal threshold : {optimal_threshold:.4f}')
    print(f'Recall at threshold: {recalls[optimal_idx]*100:.1f}%')
    print(f'Precision at threshold: {precisions[optimal_idx]*100:.1f}%')
else:
    optimal_threshold = 0.5
    print('Could not find a threshold meeting 75% recall. Using 0.5')

## 10 — Final Evaluation on All Three Splits

Now I evaluate the model with the tuned threshold across train, validation, and test sets to check for overfitting and confirm the business requirement is met.

In [ ]:
label_map = ['No Churn', 'Churn']

clf_performance(
    X_train, y_train,
    X_val, y_val,
    X_test, y_test,
    best_pipeline, label_map,
    threshold=optimal_threshold
)

## 11 — Confusion Matrix Plot (Test Set)

In [ ]:
y_proba_test = best_pipeline.predict_proba(X_test)[:, 1]
y_pred_test = (y_proba_test >= optimal_threshold).astype(int)

cm = confusion_matrix(y_test, y_pred_test)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=label_map)

fig, ax = plt.subplots(figsize=(6, 5))
disp.plot(ax=ax, colorbar=False, cmap='Blues')
ax.set_title(f'Confusion Matrix — Test Set (threshold = {optimal_threshold:.2f})',
             fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(PIPELINE_DIR, 'confusion_matrix.png'), dpi=150)
plt.show()
print('Saved: confusion_matrix.png')

## 12 — Save Classification Report

In [ ]:
report_dict = classification_report(
    y_test, y_pred_test, target_names=label_map, output_dict=True
)
report_df = pd.DataFrame(report_dict).transpose().round(2)
report_df.to_csv(os.path.join(PIPELINE_DIR, 'classification_report.csv'))
print(report_df)
print('\nSaved: classification_report.csv')

## 13 — Business Requirement Assessment

In [ ]:
churn_recall = report_dict['Churn']['recall']
churn_precision = report_dict['Churn']['precision']

print(f'Decision threshold used        : {optimal_threshold:.4f}')
print(f'Churn recall on test set       : {churn_recall*100:.1f}%')
print(f'Churn precision on test set    : {churn_precision*100:.1f}%')
print(f'Business requirement threshold : 75%')

if churn_recall >= 0.75:
    print('\n✅ BUSINESS REQUIREMENT MET: Model achieves >=75% recall on the Churn class.')
else:
    print('\n❌ BUSINESS REQUIREMENT NOT MET: Recall is below 75%.')

## 14 — Save Pipeline and Threshold

In [ ]:
pipeline_path = os.path.join(PIPELINE_DIR, 'clf_pipeline.pkl')
threshold_path = os.path.join(PIPELINE_DIR, 'optimal_threshold.pkl')

joblib.dump(best_pipeline, pipeline_path)
joblib.dump(optimal_threshold, threshold_path)

print(f'Pipeline saved to  : {pipeline_path}')
print(f'Threshold saved to : {threshold_path}')
print(f'Optimal threshold  : {optimal_threshold:.4f}')

---

## Conclusions

* A GradientBoostingClassifier pipeline was built with SMOTE oversampling and full feature preprocessing.
* Hyperparameters were searched using GridSearchCV with the CI reference parameter grid, optimising for recall on the Churn class.
* The default 0.5 decision threshold produced insufficient recall. Threshold tuning on the validation set identified a lower cutoff that meets the ≥75% recall business requirement.
* Final evaluation on the held-out test set confirms performance. Both the pipeline and the optimal threshold are saved for use in the Streamlit dashboard.